# Advanced Merging & Joins

Now that you understand the core mechanics of concatenation and merging, we can dive into professional, advanced patterns:

1.  **Exclusive Joins (Finding Discrepancies)**: Sometimes, you don't just want to merge data; you want to find **what didn't match**. For instance, you want to identify which employees *don't* have a salary record, or which products are missing from the inventory list.
2.  **Using `.join()`**: Pandas has a dedicated `df.join()` method. While `pd.merge()` is highly flexible and works with any columns, `.join()` is a specialized, high-performance method designed specifically to combine DataFrames on their **indices**.

### 2. Clear Explanation & Real-World Analogy
*   **Exclusive Joins**: Imagine a guest list at a wedding (List A) and a list of people who actually arrived (List B). Instead of listing who came, you want to find the **no-shows**. This is an exclusive join—filtering out everyone who is on *both* lists and keeping only the mismatches.
*   **`.join()`**: Think of this as gluing pages from two different books together. Book 1 has chapters 1, 2, 3. Book 2 has chapters 1, 2, 3. Because they are already structured and sorted by the chapter numbers (index), you can instantly glue them together page-by-page without searching.

#### A) Exclusive Joins using `indicator=True`
To find exclusive rows, perform an outer merge with `indicator=True`. This adds a special column named `_merge` that tells you exactly where the row came from (`left_only`, `right_only`, or `both`):


In [1]:
# Find exclusive rows
import pandas as pd

# Left DataFrame: Roles
employees = pd.DataFrame({
    'EmployeeID': [101, 102, 103, 104],
    'Name': ['Alice', 'Bob', 'Charlie', 'David'],
    'Role': ['TypeScript Dev', 'Python Dev', 'Data Analyst', 'UI Designer']
})

# Right DataFrame: Salaries (Notice EmployeeID 105 is new, and 104 is missing)
salaries = pd.DataFrame({
    'EmployeeID': [101, 102, 103, 105],
    'Salary': [85000, 95000, 90000, 120000]
})

merged_ind = pd.merge(employees, salaries, on='EmployeeID', how='outer', indicator=True)
print("--- Merged with Indicator ---")
print(merged_ind)

# Filter for rows present ONLY in the left DataFrame (unpaid employees)
unpaid_employees = merged_ind.query("_merge == 'left_only'")
print("--- Unpaid Employees ---")
print(unpaid_employees)

--- Merged with Indicator ---
   EmployeeID     Name            Role    Salary      _merge
0         101    Alice  TypeScript Dev   85000.0        both
1         102      Bob      Python Dev   95000.0        both
2         103  Charlie    Data Analyst   90000.0        both
3         104    David     UI Designer       NaN   left_only
4         105      NaN             NaN  120000.0  right_only
--- Unpaid Employees ---
   EmployeeID   Name         Role  Salary     _merge
3         104  David  UI Designer     NaN  left_only


#### B) Using `.join()` (Index-Based combining)
When your tables are already indexed by the key you want to combine on, `.join()` is much faster and cleaner than `pd.merge()`:

In [2]:
# Set index of both DataFrames to EmployeeID
employees_idx = employees.set_index('EmployeeID')
salaries_idx = salaries.set_index('EmployeeID')

# Join them using .join() (requires lsuffix/rsuffix if column names clash)
joined_df = employees_idx.join(salaries_idx, how='inner')
print(joined_df)

               Name            Role  Salary
EmployeeID                                 
101           Alice  TypeScript Dev   85000
102             Bob      Python Dev   95000
103         Charlie    Data Analyst   90000


### Common Pitfalls to Avoid
1.  **Clashing Columns with `.join()`**: Unlike `pd.merge()`, which automatically handles clashing column names, `.join()` will throw an error immediately if columns overlap and you do not provide `lsuffix` and `rsuffix` parameters.
2.  **Losing Data on Shallow Copies**: When creating copies of your DataFrames to test combinations, remember to use `df.copy()`. A simple assignment like `df2 = df` merely points to the same memory space; altering `df2` will corrupt your original data!

#### Exercise 1 (Hard)
Find all salaries in the `salaries` DataFrame that do *not* correspond to any active employee in the `employees` DataFrame using an exclusive join.

In [3]:
# Perform an outer merge with indicator
exclusive_merge = pd.merge(employees, salaries, on='EmployeeID', how='outer', indicator=True)

# Query for rows that are right_only (present in salaries but not in employees)
orphan_salaries = exclusive_merge.query("_merge == 'right_only'")
print(orphan_salaries)

   EmployeeID Name Role    Salary      _merge
4         105  NaN  NaN  120000.0  right_only


*Explanation:* Employee ID `105` is present in the `salaries` DataFrame, but does not correspond to any name or role in the `employees` DataFrame. Using `how='outer'` and filtering for `_merge == 'right_only'` lets us locate these orphaned records instantly!
